In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["langchain-session-2-mcp_server.py"],
            }
    }
)

In [5]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [6]:
tools

[StructuredTool(name='search_web', description='Search the web for information', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_webArguments', 'type': 'object'}, handle_tool_error=<function _handle_mcp_tool_error at 0x747f66f7c400>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x747f66f7f1a0>)]

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [9]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='46ac134b-1e48-4e1e-940b-57114f57c093'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 271, 'total_tokens': 428, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQxQWY3MYrkuprZut9mTRefRgsIk6', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c9e5-b8fa-7af0-8833-82cf987c4a58-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain-mcp-adapter

In [10]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [13]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
    )

In [14]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='7378fd95-54c3-4645-8df1-78de82d436fd'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 802, 'prompt_tokens': 2693, 'total_tokens': 3495, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQxdAX3heA6dAAaByZosfh1tQ8hE1', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0c9f1-afa1-7f82-b106-9690d31f056c-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom':

In [15]:
print(response["messages"][-1].content)

Here are direct (non-stop) options for San Francisco to Tokyo on March 31, 2027. All are one-way flights.

| Route | Times (Local) | Cabin | Price | Booking |
|---|---|---|---|---|
| SFO → NRT | 12:20 31 Mar 2027 → 15:25 01 Apr 2027 (11h 05m) | Economy | 471 USD | https://kiwi.com/u/rpusv9 |
| SFO → HND | 01:45 31 Mar 2027 → 05:00 01 Apr 2027 (11h 15m) | Economy | 1337 USD | https://kiwi.com/u/5q2pd2 |
| SFO → NRT | 12:20 31 Mar 2027 → 15:25 01 Apr 2027 (11h 05m) | Economy | 1497 USD | https://kiwi.com/u/bjcjnnx |
| SFO → NRT | 11:40 31 Mar 2027 → 15:00 01 Apr 2027 (11h 20m) | Economy | 1497 USD | https://kiwi.com/u/uxs7s3 |
| SFO → HND | 10:25 31 Mar 2027 → 13:55 01 Apr 2027 (11h 30m) | Economy | 1553 USD | https://kiwi.com/u/vs3fp5 |

Best price: 471 USD (NH7 SFO → NRT, 12:20 Mar 31 → 15:25 Apr 1). Shortest option: 11h 05m (NH7 SFO → NRT, 12:20 Mar 31).

Recommendation: If you want the cheapest direct option, book NH7 from SFO to NRT. If you prefer a Haneda option, NH107/UA837-style 